# 1 流式调用
举例:

In [1]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

# 1、读取.env配置文件中的信息。相关环境变量以.env文件中的优先

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
model = init_chat_model(
    model="deepseek:deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
)

In [2]:
for chunk in model.stream("帮我解释一下什么是人工智能"):
    print(chunk.text, end="", flush=True)

人工智能（Artificial Intelligence，简称 **AI**）可以简单理解为：**让机器表现出类似人类智能的能力，去完成原本需要人来做的事情**。

它不是一个单一技术，而是一大类理论、算法和应用系统的总称。核心目标包括：

- **感知**：看懂图像、听懂语音、识别文字
- **理解与推理**：理解语言、分析问题、做判断
- **学习**：从数据或经验中改进表现
- **决策**：下棋、规划路线、控制机器人
- **交互**：聊天、翻译、回答问题、生成内容

### AI 和普通程序有什么区别？
普通程序通常靠人写死规则：如果发生 A，就执行 B。  
AI 更像是在**从大量数据中找规律**。比如给 AI 看很多猫和狗的图片，它自己学会区分特征，然后判断一张新图片是猫还是狗。现在的大语言模型，比如 ChatGPT，也是通过阅读海量文本，学会预测和生成语言。

### AI 大致怎么分类？
- **弱人工智能 / 专用 AI**：只擅长某一类任务，比如人脸识别、推荐算法、语音助手。目前绝大多数 AI 都属于这一类。
- **强人工智能 / 通用 AI**：像人一样能跨领域思考和学习，目前还没有实现。
- **按方法分**：机器学习、深度学习、强化学习、符号推理等。
- **按功能分**：判别式 AI，比如判断垃圾邮件；生成式 AI，比如写文章、画图、做视频。

### AI 能做什么？
常见应用包括：搜索引擎、短视频推荐、语音助手、机器翻译、人脸识别、自动驾驶、医疗影像分析、智能客服、ChatGPT、AI 绘画等。

### AI 的局限
AI 很强大，但它通常**没有意识、情感和真正的理解**。它更像是高级的模式识别和统计预测工具，因此可能会：
- 犯错，比如“幻觉”出不存在的答案
- 带有偏见，因为训练数据有偏见
- 缺乏可解释性，有时说不清为什么这样判断
- 被滥用，涉及隐私、安全、版权和伦理问题

总结一句话：  
**人工智能是让机器通过数据和算法获得感知、学习、推理和决策能力的技术集合。它现在很强大，但本质上仍是工具，不是有意识的人。**

# 2、批量调用（batch）
举例1： 一次性接收所有的响应

In [3]:
messages = [
    "你好，你是谁？",
    "2 + 3 * 5 = ?",
    "中国首都在哪里？"
]
responses = model.batch(messages)
for response in responses:
    print(response.content)


你好！我是 DeepSeek，一个由深度求索公司开发的 AI 助手。我可以帮你回答问题、写作、翻译、分析信息，或者陪你聊天。有什么我可以帮你的吗？
17
中国的首都是**北京**。


举例2： 按照完成的顺序接收响应

In [4]:
messages = [
    "你好，你是谁？",
    "2 + 3 * 5 = ?",
    "中国首都在哪里？"
]
responses = model.batch_as_completed(messages)
for response in responses:
    print(response)

(2, AIMessage(content='中国首都是**北京**。', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户中文问题：“中国首都在哪里？” 直接回答北京。需要简洁。注意可能有些陷阱？ 中国首都是北京。可以回答：中国首都是北京。'}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 35, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 38, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'cf7a801a-7f65-4e7c-af27-ae06a3fd2539', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0f545-62df-7062-87df-2aff68e8a8f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 35, 'output_tokens': 46, 'total_tokens': 81, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reaso

举例3：性能对比

In [5]:
# 准备多个输入
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]
# ✅ 批量调用（高效）
import time

start = time.time()
responses = model.batch(inputs)
batch_time = time.time() - start

print("批量调用结果：")
for i, response in enumerate(responses):
    print(f"{i + 1}. {response.content}")
print(f"耗时: {batch_time:.2f}秒\n")

批量调用结果：
1. Spring has come.
2. Summer is very hot.
3. Autumn leaves
4. It snows in winter.
耗时: 1.99秒



作为对比，使用循环调用invoke()

In [6]:
# ❌ 循环调用（低效，仅用于对比）
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]

start = time.time()
loop_responses = []
for inp in inputs:
    response = model.invoke(inp)
    loop_responses.append(response)
loop_time = time.time() - start

for i, response in enumerate(responses):
    print(f"{i + 1}. {response.content}")

print(f"循环调用耗时: {loop_time:.2f}秒")

print(f"批量调用节省: {((loop_time - batch_time) / loop_time * 100):.1f}%")

1. Spring has come.
2. Summer is very hot.
3. Autumn leaves
4. It snows in winter.
循环调用耗时: 4.76秒
批量调用节省: 58.3%
